# Copilot Org Data Direct Ingester (Databricks)

Databricks adaptation of the corresponding ValueLens notebook. It uses the configured Unity Catalog schema and volume; see the [Databricks route guide](../README.md) for setup and scheduling.


In [ ]:
from valuelens_databricks_runtime import initialize
VL = initialize(spark, dbutils)
DB_SCHEMA = VL.schema
VL.record_task_start()


In [ ]:
# === CONFIG ===
TENANT_ID = VL.widget("tenant_id", "")
CLIENT_ID = VL.widget("client_id", "")
CLIENT_SECRET = VL.secret()

OUTPUT_TABLE  = f'{DB_SCHEMA}.copilot_org_data'
WRITE_MODE    = 'overwrite'
ALLOW_EMPTY_SNAPSHOT = False   # set True only for an intentional empty first install


In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')

import time

GRAPH_MAX_ATTEMPTS = 6     # one try plus up to five retries
GRAPH_MAX_WAIT_SEC = 120   # longest single wait, even if Graph asks for more


def _graph_wait_seconds(response, attempt):
    try:
        wait = float(response.headers.get('Retry-After', ''))
    except (TypeError, ValueError):
        wait = 5 * 2 ** attempt
    return min(max(wait, 1.0), GRAPH_MAX_WAIT_SEC)


def graph_get(url, timeout=60):
    """GET from Graph. Waits out throttling (429) and brief outages (5xx, dropped connections),
    and gets a new token once if the old one expires partway through a long pull."""
    global token, headers
    renewed = False
    for attempt in range(1, GRAPH_MAX_ATTEMPTS + 1):
        last = attempt == GRAPH_MAX_ATTEMPTS
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
        except (requests.ConnectionError, requests.Timeout) as exc:
            if last:
                raise
            wait = min(5 * 2 ** attempt, GRAPH_MAX_WAIT_SEC)
            print(f'  Graph request failed ({type(exc).__name__}); trying again in {wait}s.')
            time.sleep(wait)
            continue
        if r.status_code == 401 and not renewed and not last:
            token = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
            headers = {'Authorization': f'Bearer {token}'}
            renewed = True
            continue
        if (r.status_code == 429 or r.status_code >= 500) and not last:
            wait = _graph_wait_seconds(r, attempt)
            print(f'  Graph returned {r.status_code}; trying again in {wait:.0f}s.')
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r


In [ ]:
select_fields  = 'userPrincipalName,displayName,department,jobTitle,companyName,officeLocation,city,country,accountEnabled'
expand_manager = 'manager($select=userPrincipalName)'
next_url = f'https://graph.microsoft.com/v1.0/users?$select={select_fields}&$expand={expand_manager}&$top=999'


def _validate_users_page(data, page_number):
    if not isinstance(data, dict):
        raise ValueError(f'Graph /users page {page_number} did not return an object.')
    if 'value' not in data:
        raise ValueError(f"Graph /users page {page_number} is missing required 'value'; refusing to overwrite org data.")
    value = data['value']
    if not isinstance(value, list):
        raise ValueError(f"Graph /users page {page_number} returned a non-list 'value'; refusing to overwrite org data.")
    next_link = data.get('@odata.nextLink')
    if next_link is not None and (not isinstance(next_link, str) or not next_link.strip()):
        raise ValueError(f"Graph /users page {page_number} returned an invalid '@odata.nextLink'.")
    for item_number, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'Graph /users page {page_number} item {item_number} is not an object.')
    return value, next_link


users = []
seen_urls = set()
page_number = 0
while next_url:
    if next_url in seen_urls:
        raise ValueError(f'Graph /users paging loop detected at page {page_number + 1}.')
    seen_urls.add(next_url)
    page_number += 1
    r = graph_get(next_url, timeout=60)
    page, next_url = _validate_users_page(r.json(), page_number)
    users.extend(page)
    print(f'  Fetched {len(users):,} users so far...')

print(f'✓ Total users: {len(users):,}')


In [ ]:
def _normalise_identity(value):
    return (value or '').strip().lower()


def _canonical_org_row(user):
    if not isinstance(user, dict):
        raise ValueError('Org data item is not an object.')
    norm = _normalise_identity(user.get('userPrincipalName'))
    if not norm:
        raise ValueError('Org data row is missing userPrincipalName; refusing to write an anonymous identity.')
    manager = user.get('manager')
    manager_upn = ''
    if manager not in (None, ''):
        if not isinstance(manager, dict):
            raise ValueError(f'Org data row {norm!r} has a non-object manager payload.')
        manager_upn = manager.get('userPrincipalName', '') or ''
    return {
        'PersonId':         user.get('userPrincipalName'),
        'displayName':      user.get('displayName'),
        'Organization':     user.get('department'),
        'JobTitle':         user.get('jobTitle'),
        'companyName':      user.get('companyName'),
        'officeLocation':   user.get('officeLocation'),
        'city':             user.get('city'),
        'country':          user.get('country'),
        'accountEnabled':   str(user.get('accountEnabled', '')),
        'managerUPN':       manager_upn,
    }


def _dedupe_org_rows(rows):
    seen = {}
    deduped = []
    for row in rows:
        norm = _normalise_identity(row.get('PersonId'))
        comparable = {key: '' if value is None else str(value).strip() for key, value in row.items()}
        prior = seen.get(norm)
        if prior is None:
            seen[norm] = comparable
            deduped.append(row)
            continue
        if comparable != prior:
            raise ValueError(f'Conflicting org rows detected for {norm!r}; refusing to overwrite a good snapshot.')
    return deduped


rows = _dedupe_org_rows([_canonical_org_row(user) for user in users])
print(f'Built {len(rows):,} canonicalised rows.')


In [ ]:
# Deepest LevelN column to emit. Mirrors the PAX default; harmless if the org
# is shallower (extra LevelN columns just come back empty).
MAX_ORG_LEVELS = 14

HIER_FIXED   = ['OrgLevel', 'HierarchyPath', 'TopOfChain_Name', 'IsManager', 'DirectReports', 'HierarchyError',
                'OrgPath']
HIER_LEVELS  = [f'Level{i}_Name' for i in range(MAX_ORG_LEVELS + 1)]
HIER_COLUMNS = HIER_FIXED + HIER_LEVELS


def build_hierarchy(rows, max_levels=MAX_ORG_LEVELS):
    """Flatten each person's managerUPN chain into Level0..N + org metadata.

    Cycle-tolerant: a cycle truncates the chain and is recorded in HierarchyError.
    Someone who is their own manager is the top of their chain, not an error.
    OrgPath is the chain as lower-case UPNs joined by '|' (top first), the form DAX PATHCONTAINS reads.
    """
    def _n(v):
        return (v or '').strip().lower()

    name_of, mgr_of = {}, {}
    for row in rows:
        user = _n(row.get('PersonId'))
        if not user:
            continue
        name_of[user] = (row.get('displayName') or row.get('PersonId') or '').strip()
        mgr_of[user] = _n(row.get('managerUPN'))

    direct = {user: 0 for user in name_of}
    for user, manager in mgr_of.items():
        if manager and manager != user and manager in direct:
            direct[manager] += 1

    out = {}
    for user in name_of:
        chain, seen_chain, cur, error = [], set(), user, ''
        while cur and cur in name_of:
            if cur in seen_chain:
                error = f'Cycle detected at {cur!r}'
                break
            seen_chain.add(cur)
            chain.append(cur)
            manager = mgr_of.get(cur, '')
            cur = '' if manager == cur else manager
        chain = list(reversed(chain))
        rec = {column: '' for column in HIER_COLUMNS}
        for idx in range(max_levels + 1):
            rec[f'Level{idx}_Name'] = name_of.get(chain[idx], '') if idx < len(chain) else ''
        rec['OrgLevel'] = str(max(len(chain) - 1, 0))
        rec['HierarchyPath'] = ' > '.join(name_of.get(node, node) for node in chain)
        rec['TopOfChain_Name'] = name_of.get(chain[0], '') if chain else ''
        rec['IsManager'] = 'TRUE' if direct.get(user, 0) > 0 else 'FALSE'
        rec['DirectReports'] = str(direct.get(user, 0))
        rec['HierarchyError'] = error
        rec['OrgPath'] = '|'.join(chain)
        out[user] = rec
    return out


_hier = build_hierarchy(rows, MAX_ORG_LEVELS)
_blank = {column: '' for column in HIER_COLUMNS}
for row in rows:
    user = (row.get('PersonId') or '').strip().lower()
    row.update(_hier.get(user, _blank))

_mgrs = sum(1 for row in rows if row.get('IsManager') == 'TRUE')
_depth = max((int(row.get('OrgLevel') or 0) for row in rows), default=0)
print(f'Built manager hierarchy for {len(_hier):,} people | managers: {_mgrs:,} | max depth: {_depth}')
_cycles = sorted({rec['HierarchyError'] for rec in _hier.values() if rec['HierarchyError']})
if _cycles:
    print(f'Manager cycles: {len(_cycles):,} distinct (rows kept, chain truncated). Examples:')
    for _error in _cycles[:20]:
        print(f'  {_error}')

# How many people have a department. Views by department leave out everyone without one.
LOW_DEPARTMENT_COVERAGE = 0.8


def department_coverage(departments):
    """(people with a department, people) for the `Organization` values of an org table."""
    values = list(departments)
    have = sum(1 for v in values if v is not None and str(v).strip())
    return have, len(values)


def coverage_message(have, total):
    """One line on department coverage, with advice when views by department would leave many out."""
    line = f'{have:,} of {total:,} people have a department'
    if not total or have / total >= LOW_DEPARTMENT_COVERAGE:
        return line
    return (f'{line}. Views by department leave out everyone without one: fill in Department on users '
            'in Microsoft Entra ID.')


_have_dept, _people = department_coverage(row.get('Organization') for row in rows)
print(('✓ ' if not _people or _have_dept / _people >= LOW_DEPARTMENT_COVERAGE else '⚠ ')
      + coverage_message(_have_dept, _people))


In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _guard_org_snapshot(row_count, table_name):
    exists = _table_exists(table_name)
    if row_count == 0 and exists:
        raise ValueError(f'Fetched 0 org rows; refusing to replace existing {table_name}.')
    if row_count == 0 and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'Fetched 0 org rows and {table_name} does not exist yet. '
            'Set ALLOW_EMPTY_SNAPSHOT = True only for an intentional empty first install.'
        )


schema = StructType([
    StructField('PersonId',        StringType(), True),
    StructField('displayName',     StringType(), True),
    StructField('Organization',    StringType(), True),
    StructField('JobTitle',        StringType(), True),
    StructField('companyName',     StringType(), True),
    StructField('officeLocation',  StringType(), True),
    StructField('city',            StringType(), True),
    StructField('country',         StringType(), True),
    StructField('accountEnabled',  StringType(), True),
    StructField('managerUPN',      StringType(), True),
])
for column in HIER_COLUMNS:
    schema = schema.add(StructField(column, StringType(), True))

df = spark.createDataFrame(rows, schema=schema)
df = df.withColumn(
    'PersonId_Normalized',
    F.when(F.col('PersonId').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('PersonId').cast('string'))))
)

duplicate_people = (
    df.filter(F.col('PersonId_Normalized').isNotNull() & (F.col('PersonId_Normalized') != ''))
      .groupBy('PersonId_Normalized').count()
      .filter(F.col('count') > 1)
      .count()
)
if duplicate_people:
    raise ValueError('Org data still contains duplicate PersonId_Normalized values after validation.')

total = df.count()
_guard_org_snapshot(total, OUTPUT_TABLE)
df = df.withColumn('TotalEmployees', F.lit(str(total)))

_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'Final rows: {total:,}')
print('Columns:', df.columns)


In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('PersonId', 'PersonId_Normalized', 'Organization', 'JobTitle', 'managerUPN').show(10, truncate=False)
tbl.groupBy('Organization').count().orderBy(F.desc('count')).show(20, truncate=False)
tbl.select('PersonId', 'displayName', 'managerUPN', 'OrgLevel', 'IsManager', 'DirectReports', 'HierarchyPath').show(10, truncate=False)

VL.record_task_success()
